## FormalVizWidget

A generic `anywidget` for animating an SVG whose visual state is driven by
a formal model. This notebook is a reusable framework for implementing animation for the spectabular specifcation libary. A
model specific example notebook is given in `elevator/Elevator.ipynb`.

To use, import the notebook with the `%run` shell magic

```
%run "../FormalVizWidget.ipynb"
```

and then supply:

- an SVG with element `id`s to animate,
- a `state` dict shape,
- `widget[key] = (fn, method)` bindings mapping state to element
  properties (see "Binding SVG properties to state" below),
- a Spectabular relation spec, as a class variable on a `FormalVizWidget`
  subclass.

In [1]:
import asyncio
import json
import os
import sys
import time as _time
import xml.etree.ElementTree as ET

import anywidget
import traitlets

ET.register_namespace("", "http://www.w3.org/2000/svg")

# anim_algebra.py lives alongside this notebook. %run resolves relative to
# the calling notebook's directory (e.g. elevator/, automotive/), so find
# anim_algebra.py explicitly rather than relying on cwd.
for _candidate in (os.getcwd(), os.path.join(os.getcwd(), "..")):
    if os.path.exists(os.path.join(_candidate, "anim_algebra.py")) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from anim_algebra import (
    Animation, Leaf, E, is_identity, seq, svg_generator, svg_diff, svg_apply, compile_to_svg,
)


### The widget

`FormalVizWidget` renders `_svg_content` into the DOM once, then applies
`widget[key] = (fn, method)` bindings to the live DOM every time the
traitlets `state` dict changes: Python evaluates each binding's `fn`
against the new `state` and syncs only the resulting values, and the JS
side interpolates between the old and new values using the `tween`/`snap`
helpers.

`state` is just what's currently on screen. Each widget instance also
keeps its own local, Python-only history of how it got there
(`_history`/`_events`/`_cursor`) so that:

- `w.snapshot()` can fork off a brand new, independent widget (its own
  model, its own DOM) from wherever `w` currently is.
- `w.replay()`, `w.export_trace()`, `w.goto_transition(i)`, `w.prev()`,
  `w.next()` can navigate or dump that history without re-solving the
  spec, and without disturbing that history's ability to differ between
  `w` and any widget forked from it.

In [2]:
ESM = r"""
export default {
  render({ model, el }) {
    const container = document.createElement("div");
    el.appendChild(container);

    function renderSVG() {
      container.innerHTML = model.get("_svg_content");
      const svg = container.querySelector("svg");
      if (!svg) return;
      const vb = model.get("_viewbox");
      if (vb) svg.setAttribute("viewBox", vb);
    }

    model.on("change:_svg_content", renderSVG);
    renderSVG();
  }
};
"""


In [3]:
class FormalVizWidget(anywidget.AnyWidget):
    _esm = ESM

    _svg_content = traitlets.Unicode("").tag(sync=True)
    _viewbox = traitlets.Unicode("").tag(sync=True)
    _max_width = traitlets.Unicode("800px").tag(sync=True)

    # the displayed state, mirrored to JS. always equal to
    # self._states[self._cursor]
    state = traitlets.Dict({}).tag(sync=True)

    # set by a subclass to a Spectabular relation spec: a `VectorTable`, or
    # a `Table` selecting among named relations, or
    # several combined with operators.
    # basically what people would expect
    spec = None

    def __init__(self, svg_content="", variables=None,
                 viewbox=None, max_width="800px", **kw):
        initial = dict(variables or {})
        # pristine original SVG. never mutated -- svg_generator/svg_diff/
        # svg_apply/compile_to_svg all render fresh against this, so
        # _svg_content (below) is always free to just be "whatever's on
        # screen right now" instead of "the untouched original".
        self._base_svg = svg_content
        # "selector.prop" -> fn(state) -> value. python-only, can't be a
        # traitlet since it holds closures. tween-vs-snap is never
        # specified here: it's inferred per transition by diffing rendered
        # frames (see anim_algebra.svg_diff), not hand-tagged.
        self._recipe = {}
        super().__init__(
            _svg_content=svg_content,
            state=dict(initial),
            _viewbox=viewbox or "",
            _max_width=max_width,
            **kw,  # allow user to edit anywidget settings
        )
        self._initial = initial
        # states visited: _states[i] is the state after _events[i-1] fired
        # (_states[0] is always the initial state, so
        # len(_states) == len(_events) + 1). _cursor indexes into _states
        # for whichever point is currently on screen, normally the most
        # recent entry, but goto_transition/prev/next can move it back
        # without discarding anything, so scrubbing through history
        # doesn't require re-running the formal model.
        self._states = [dict(initial)]
        self._events = []
        # seconds per recorded transition; self._durations[i] is how long
        # the step _states[i] -> _states[i+1] took. Only needed for static
        # export (_static_svg); the live widget always bakes its duration
        # directly into _svg_content's SMIL markup per transition.
        self._durations = []
        # one Animation per recorded transition (anim_algebra.svg_diff's
        # output): _history[i] transitions _states[i] -> _states[i+1].
        # this *is* the algebra's own record of what happened -- a ledger
        # for w.history/export_trace/static export, decoupled from live
        # navigation (goto_transition/prev/next), which always re-diffs
        # fresh rather than replaying/inverting recorded entries.
        self._history = []
        self._cursor = 0

    def _repr_mimebundle_(self, **kwargs):
        data, metadata = super()._repr_mimebundle_(**kwargs)
        data["image/svg+xml"] = self._static_svg()
        return data, metadata

    def _static_svg(self):
        # a static, standalone SVG replaying the whole displayed trace
        # (up to the cursor) via SMIL, for export (compile.sh/PDF) or a
        # SMIL-blind consumer. seq() silently drops identity animations
        # during normalization, so steps that produced no visible diff
        # (e.g. a Tick event that touches nothing bound) are filtered out
        # here first -- otherwise zipping per-step durations against
        # Seq.children would drift out of alignment.
        pairs = [
            (a, d) for a, d in zip(self._history[:self._cursor], self._durations[:self._cursor])
            if not is_identity(a)
        ]
        if not pairs:
            return svg_generator(self._states[self._cursor], self._recipe, self._base_svg)
        anims, durs = zip(*pairs)
        return compile_to_svg(seq(*anims), self._base_svg, leaf_dur=list(durs))

    @classmethod
    def from_svg(cls, path, variables, viewbox=None, **kw):
        with open(path) as f:
            svg = f.read()
        return cls(svg, variables, viewbox, **kw)

    @property
    def s(self):
        return _S(self)  # delegate

    # --- declarative SVG bindings -------------------------------------
    #
    # widget["selector.prop"] = fn
    #   fn      : state -> value. called with the widget's current state
    #             whenever a transition is displayed.
    #   prop    : "text" sets el.textContent instead of an attribute;
    #             "attr:name" sets the raw SVG attribute "name" via
    #             setAttribute instead. Both are applied instantly,
    #             since neither has a SMIL animation equivalent (SMIL only
    #             animates presentation attributes, not text content).
    #             "transform" is special: fn must return a structured op
    #             `(kind, args)` instead of a plain value, since
    #             <animateTransform> needs a fixed `type` plus bare
    #             numeric args rather than a CSS transform string, e.g.
    #             `("translate", (x, y))`, `("rotate", (deg, cx, cy))`, or
    #             `("scale", (sx, sy))`. Every other prop is a plain SVG
    #             presentation attribute (`fill`, `opacity`, ...), set
    #             directly via setAttribute.
    #
    # tween-vs-snap is never hand-specified: anim_algebra.svg_diff infers
    # it per transition by comparing the rendered frame before and after,
    # so there's no method to pick here -- just fn.

    def __setitem__(self, key, fn):
        if not callable(fn):
            raise TypeError(
                f"widget[{key!r}] = fn: fn(state) -> value. tween-vs-snap "
                "is inferred per transition by diffing rendered frames "
            )
        _parse_binding_key(key)
        self._recipe[key] = fn
        self._svg_content = svg_generator(self._states[self._cursor], self._recipe, self._base_svg)

    def __getitem__(self, key):
        return self._recipe[key]

    def __delitem__(self, key):
        del self._recipe[key]
        self._svg_content = svg_generator(self._states[self._cursor], self._recipe, self._base_svg)

    def __contains__(self, key):
        return key in self._recipe

    # --------------------------------------------------------------------

    def _record(self, next_state, event, label, duration):
        # if the cursor isn't at the head (the user rewound with prev()/
        # goto_transition() and is now doing something new), the
        # abandoned future is discarded here
        del self._states[self._cursor + 1:]
        del self._events[self._cursor:]
        del self._durations[self._cursor:]
        del self._history[self._cursor:]
        frame_a = svg_generator(self._states[self._cursor], self._recipe, self._base_svg)
        frame_b = svg_generator(next_state, self._recipe, self._base_svg)
        anim = svg_diff(frame_a, frame_b)
        self._events.append(event if event is not None else label)
        self._states.append(dict(next_state))
        self._durations.append(duration)
        self._history.append(anim)
        self._cursor = len(self._states) - 1
        return frame_a, frame_b, anim

    def _display(self, frame_a, frame_b, anim, next_state, duration):
        self._svg_content = svg_apply(anim, frame_a, dur=duration) if duration > 0 else frame_b
        self.state = dict(next_state)

    def reset(self):
        # back to the initial state, and forget the history that led
        # anywhere else
        self._states = [dict(self._initial)]
        self._events = []
        self._durations = []
        self._history = []
        self._cursor = 0
        self._svg_content = svg_generator(self._initial, self._recipe, self._base_svg)
        self.state = dict(self._initial)

    async def transition(self, next_state, duration=0.3, event=None, label=None):
        # await until transition
        frame_a, frame_b, anim = self._record(next_state, event, label, duration)
        self._display(frame_a, frame_b, anim, next_state, duration)
        await asyncio.sleep(duration)

    def transition_sync(self, next_state, duration=0.3, event=None, label=None):
        # block until transition
        frame_a, frame_b, anim = self._record(next_state, event, label, duration)
        self._display(frame_a, frame_b, anim, next_state, duration)
        _time.sleep(duration)

    def snap(self, next_state, event=None, label=None):
        frame_a, frame_b, anim = self._record(next_state, event, label, 0.0)
        self._display(frame_a, frame_b, anim, next_state, 0)

    # move the display around in already-recorded history. non-animated
    # by default since these are typically single scrub steps run from
    # their own cell; pass duration for a tween instead. Always re-diffs
    # the two frames fresh rather than replaying self._history[index] --
    # a recorded step's Animation is forward-only (states[i]->states[i+1]),
    # so e.g. prev() would need its inverse, which is simplest to get by
    # just re-diffing swapped frames rather than inverting a Leaf.

    def goto_transition(self, index, duration=0.0):
        index = max(0, min(index, len(self._states) - 1))
        if duration > 0:
            frame_a = svg_generator(self._states[self._cursor], self._recipe, self._base_svg)
            frame_b = svg_generator(self._states[index], self._recipe, self._base_svg)
            self._svg_content = svg_apply(svg_diff(frame_a, frame_b), frame_a, dur=duration)
        else:
            self._svg_content = svg_generator(self._states[index], self._recipe, self._base_svg)
        self._cursor = index
        self.state = dict(self._states[index])

    def prev(self, duration=0.0):
        if self._cursor > 0:
            self.goto_transition(self._cursor - 1, duration)

    def next(self, duration=0.0):
        if self._cursor < len(self._states) - 1:
            self.goto_transition(self._cursor + 1, duration)

    @property
    def cursor(self):
        return self._cursor

    @property
    def can_prev(self):
        return self._cursor > 0

    @property
    def can_next(self):
        return self._cursor < len(self._states) - 1

    @property
    def history(self):
        # one Animation per recorded transition (anim_algebra.svg_diff's
        # output): history[i] transitions states[i] -> states[i+1]
        return list(self._history)

    @property
    def states(self):
        # states visited, index 0 is always the initial state
        return [dict(s) for s in self._states]

    @property
    def durations(self):
        # seconds per recorded transition, durations[i] is states[i] -> states[i+1]
        return list(self._durations)

    def __len__(self):
        # number of recorded transitions (== len(states) - 1)
        return len(self._events)

    async def replay(self, tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None):
        """
        Re-animate through already-recorded history, in order,
        without recomputing states
        """
        if to_index is None:
            to_index = len(self._states) - 1
        self.goto_transition(from_index, duration=0)
        for i in range(from_index + 1, to_index + 1):
            self.goto_transition(i, duration=transition_dur)
            await asyncio.sleep(transition_dur)
            await asyncio.sleep(max(0.0, tick_dt - transition_dur))

    def export_trace(self, as_json=False):
        trace = [
            {
                "index": i,
                "event": self._events[i],
                "before": self._states[i],
                "after": self._states[i + 1],
                "duration": self._durations[i],
            }
            for i in range(len(self._events))
        ]
        return json.dumps(trace, default=str) if as_json else trace

    def snapshot(self):
        """
        A new, fully independent widget (its own model and DOM) showing
        this widget's current displayed state. Carries along the history up
        to the cursor.
        """
        cls = type(self)
        copy = cls(
            svg_content=self._base_svg,
            variables=self._states[self._cursor],
            viewbox=self._viewbox,
            max_width=self._max_width,
        )
        copy._recipe = dict(self._recipe)
        copy._states = [dict(s) for s in self._states[:self._cursor + 1]]
        copy._events = list(self._events[:self._cursor])
        copy._durations = list(self._durations[:self._cursor])
        copy._history = list(self._history[:self._cursor])
        copy._cursor = self._cursor
        copy._svg_content = svg_generator(copy._states[copy._cursor], copy._recipe, copy._base_svg)
        return copy

    @classmethod
    def _flattened_spec(cls):
        """
        `cls.spec`, flattened and indexed by free variable name, cached on
        the concrete subclass.
        had to use cls.__dict__ instead of hasattr to prevent shadowing from parent
        """
        if "_flat_spec_cache" not in cls.__dict__:
            if cls.spec is None:
                raise NotImplementedError(f"{cls.__name__}.spec is not set")
            flat = flatten(cls.spec)
            free_by_name = {str(v): v for v in freevars(flat)}
            if isinstance(cls.spec, VectorTable):
                # The variables the pecs assigns values to, one per row of
                # .left. primed (a relation table's next state values) or
                # unprimed (a predicate table's values).
                output_names = {str(v) for v in cls.spec.left}
            else:
                # No `.left` to read (like a Table()).
                # a free variable ending in `ʹ` is a next-state output, everything
                # else a read-only input.
                output_names = {name for name in free_by_name if name.endswith("ʹ")}
            cls._flat_spec_cache = (flat, free_by_name, output_names)
        return cls._flat_spec_cache

    @classmethod
    def compute_next_state(cls, state, **event):
        """
        Solve `cls.spec` for the values of whatever it assigns to
        given `state` plus whatever else a particular row needs.

        Works uniformly for both roles a VectorTable can play:
        - a relation table, where `.left` holds primed (`xʹ`) next-state
          variables. This is the usual case,the result is a next state,
          keyed by the unprimed names.
        - a **predicate table**, where `.left` holds unprimed
          variables derived from the current state. Itreturns
          those derived values, keyed by their own names.

        Every other free variable in the spec is a read only input.
        We pin from `state` if present there, else from `event` if supplied.
        An unprimed variable that's the counterpart of a primed output (i.e.
        clearly meant to be persisted state) must be in `state`, or this
        raises `KeyError`. Any other input (an event selector, a
        predicate's own extra parameter, ...) is simply left unconstrained
        if missing. Harmless if the row that fires doesn't actually
        depend on it.

        A classmethod rather than an instance method (and `spec` a class
        variable rather than instance state) so it also works without a
        live widget.
        """
        flat, free_by_name, output_names = cls._flattened_spec()
        input_names = set(free_by_name) - output_names
        required_state_names = {name.removesuffix("ʹ") for name in output_names if name.endswith("ʹ")}

        solver = z3.Solver()
        solver.add(flat)
        for name in input_names:
            if name in state:
                # read any free vars from state
                value = state[name]
            elif name in event:
                # read any free vars from event
                value = event[name]
            elif name in required_state_names:
                raise KeyError(f"state is missing {name!r}, required by {cls.__name__}.spec")
            else:
                continue  # not relevant to whichever row fires; let Z3 pick freely
            var = free_by_name[name]
            solver.add(var == _state_value_to_z3(value, var.sort()))

        result = solver.check()
        if result != z3.sat:
            raise ValueError(
                f"{cls.__name__}.spec has no enabled row for {event!r} from state {state!r} ({result})"
            )
        # a table can have several satisfying assignments for its
        # output variables (nondeterminism). this returns whichever one
        # Z3 happens to find, which is fine for scripted/replayed
        # animation (the scenario already picked the transition/row via
        # `event`) but not for interactively exploring all enabled next
        # states, which would need re-solving under a blocking clause per
        # answer found.
        model = solver.model()
        return {
            name.removesuffix("ʹ"): _z3_to_state_value(model.eval(free_by_name[name], model_completion=True))
            for name in output_names
        }


def _parse_binding_key(key):
    # "selector.prop" -> ("selector", "prop"); selector is looked up via
    # getElementById first, then as a CSS selector, so plain element ids
    # (the common case) and full selectors both work as long as the
    # selector itself doesn't contain a dot.
    if not isinstance(key, str) or "." not in key:
        raise ValueError(
            f"binding key {key!r} must look like 'selector.prop', e.g. "
            "'button1up.fill' or 'floor-display.text'"
        )
    selector, prop = key.rsplit(".", 1)
    if not selector or not prop:
        raise ValueError(f"binding key {key!r} must look like 'selector.prop'")
    return selector, prop


class _S:
    # attribute access of a widget's state (w.s.blinkRight instead of w.state["blinkRight"]).

    def __init__(self, widget):
        object.__setattr__(self, "_widget", widget)

    def __getattr__(self, name):
        try:
            return self._widget.state[name]
        except KeyError:
            raise AttributeError(name) from None

    def __setattr__(self, name, value):
        # allows us to use the ** syntax. goes through snap() (not the
        # trait directly) so direct field edits still land in history.
        next_state = dict(self._widget.state)
        next_state[name] = value
        self._widget.snap(next_state, event={name: value})

    def __repr__(self):
        return repr(self._widget.state)


def _state_value_to_z3(value, sort):
    if isinstance(value, bool):
        return z3.BoolVal(value)
    if isinstance(value, int):
        return z3.IntVal(value)
    if isinstance(value, float):
        return z3.RealVal(value)
    if isinstance(value, str):
        for i in range(sort.num_constructors()):
            if sort.constructor(i).name() == value:
                return sort.constructor(i)()
        raise ValueError(f"{value!r} is not a value of enum sort {sort}")
    raise TypeError(f"don't know how to convert {value!r} (a {type(value).__name__}) to a Z3 term")


def _z3_to_state_value(value):
    kind = value.sort().kind()
    if kind == z3.Z3_BOOL_SORT:
        return z3.is_true(value)
    if kind == z3.Z3_INT_SORT:
        return value.as_long()
    if kind == z3.Z3_REAL_SORT:
        return value.as_fraction()
    return str(value)  # enum (or other datatype) constructor name


TODO:
- Expose multiple satisfying next states for a nondeterministic relation
  (re-solve per model found) to support
  interactive exploration rather than only scripted replay.
- Catch trying to use 2D Tables in the animation. Maybe we put this onus on the animator?

### Scenario runners
Helpers to run scenarios.

In [4]:
async def run_scenario(w, events, tick_dt=0.5, transition_dur=0.15):
    """
    Run a sequence of events, animating smoothly between each.

    w : FormalVizWidget
    events : list of dicts, one set of `compute_next_state` keyword arguments per event
    tick_dt : seconds to wait after each event
    transition_dur : animation duration per transition
    """
    for event in events:
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))


async def advance_ticks(w, n, event, tick_dt=0.5, transition_dur=0.15, stop_when=None):
    """
    Send the same event `n` times in a row animating between each.
    Stops early if `stop_when(state)` becomes true. Usefull for
    modeling time based applications

    Returns the number of events actually sent.
    """
    for i in range(n):
        if stop_when is not None and stop_when(w.state):
            return i
        next_state = w.compute_next_state(w.state, **event)
        await w.transition(next_state, duration=transition_dur, event=event)
        await asyncio.sleep(max(0.0, tick_dt - transition_dur))
    return n


### General helpers
useful for debugging or testing any `compute_next_state` function against any widget.

In [5]:
def diff_state(before, next):
    # return what varibles in state change from before to next
    keys = before.keys() | next.keys()
    return {k: (before.get(k), next.get(k)) for k in keys if before.get(k) != next.get(k)}


def trace_scenario(widget_cls, initial_state, events):
    # tracer with no animation
    state = dict(initial_state)
    trace = []
    for event in events:
        next_state = widget_cls.compute_next_state(state, **event)
        trace.append((event, dict(state), dict(next_state)))
        state = next_state
    return trace


async def reset_and_run(w, scenario_fn, *args, settle=0.3, **kwargs):
    """
    Reset a widget to its initial state, let the reset render settle, then
    run `scenario_fn(w, *args, **kwargs)` (e.g. a demo_* function or
    `run_scenario`). 
    """
    w.reset()
    await asyncio.sleep(settle)
    await scenario_fn(w, *args, **kwargs)

### Using this framework

This notebook defines `FormalVizWidget` (with its generic
`compute_next_state`), `run_scenario`, `advance_ticks`, `diff_state`,
`trace_scenario`, and `reset_and_run` in the calling kernel once `%run`
has executed it. A model notebook:

1. `%run`s this notebook, then `%run`s `spectabular.ipynb` for
   `Bool`/`Int`/`EnumType`/`VectorTable`/`flatten`/...
2. builds its spec and subclasses `FormalVizWidget` with `spec = ...` set
   as a class variable.

See `elevator/Elevator.ipynb` for a full worked example.

#### Changing state

- `w.transition(next_state, duration=0.3, event=None, label=None)`.
  async, animates and records a new step in `w`'s history.
- `w.transition_sync(...)`. same, but blocks instead of `await`ing.
- `w.snap(next_state, event=None, label=None)`. instant, still recorded.
- `w.s.name = value`. shorthand for a one-field `snap`; also recorded.
- `w.reset()`. back to the initial state, discarding all history.

`event`/`label` is whatever you want attached to that step for
`export_trace()` later, typically the same kwargs dict passed to
`compute_next_state`, which is what `run_scenario`/`advance_ticks` record
automatically.

#### Forking: `snapshot()`

```python
w1 = w.snapshot()   # independent widget, currently identical to w
w.transition(...)   # only w moves
w1.transition(...)  # only w1 moves, from where it was snapshotted
```

`snapshot()` returns a new widget of the same class, showing wherever
`w`'s cursor currently is (not necessarily its latest state, see
below), carrying along the history that led there. From that point on
the two widgets never affect each other.

#### Transition history

Every visible change is driven by `anim_algebra`: `w.transition`/`snap`/
`goto_transition` render the "before" and "after" states as concrete SVG
frames (`anim_algebra.svg_generator`), diff them (`anim_algebra.svg_diff`)
to get an `Animation`, and, if animating, bake it onto the screen with
`anim_algebra.svg_apply`. Tween vs. snap is never chosen by hand — it's
whatever `svg_diff` infers per property from comparing the two frames.

- `w.history` - list of every recorded `Animation` (from
  `anim_algebra.svg_diff`), `history[i]` is the transition from
  `states[i]` to `states[i+1]`.
- `w.states` - list of every state visited, `states[0]` is the initial
  state (this is what `w.history` used to return, before it became the
  transitions themselves).
- `w.durations` - seconds each recorded transition took,
  `durations[i]` is `states[i] -> states[i+1]`.
- `w.cursor`, `len(w)` - current position, and number of recorded
  transitions.
- `w.export_trace(as_json=False)` - list (or JSON string) of
  `{index, event, before, after, duration}` per transition.
- `w.goto_transition(i, duration=0.0)` - jump the display to `states[i]`
  without re-solving anything or changing what's recorded. Diffs and
  animates fresh between the current and target frame if `duration > 0`,
  regardless of how far the jump is.
- `w.prev(duration=0.0)` / `w.next(duration=0.0)` - step the cursor by
  one; `w.can_prev` / `w.can_next` say whether there's anywhere to go.
- `await w.replay(tick_dt=0.5, transition_dur=0.15, from_index=0, to_index=None)`
  - re-animate through recorded history in order. Unlike re-running
  `run_scenario`, this never calls `compute_next_state` again, so it
  reproduces exactly what happened even if the spec is nondeterministic.

Transitioning after rewinding with `prev()`/`goto_transition()` truncates
whatever history was ahead of the cursor before recording new stuff. `w.snapshot()` can
preserve a branch point instead of overwriting it.

#### Binding SVG properties to state

For the common case, where some property of some element should just track
a function of `state`:

```python
w["cabin-group.transform"] = lambda s: ("translate", (0, FLOOR_Y[s["floor"]] - FLOOR_Y["F1"]))
w["btn-b1u.fill"] = lambda s: "#ff9800" if s["b1u"] else "#bdbdbd"
w["floor-display.text"] = lambda s: s["floor"]
```

`widget[key] = fn`:

- `key` is `"selector.prop"`. `selector` is looked up with
  `getElementById` first, then as a CSS selector, so a plain element id
  (the usual case) and a full selector both work, as long as the
  selector itself doesn't contain a dot. `prop` is normally a plain SVG
  presentation attribute (`fill`, `opacity`, ...), set via
  `setAttribute` so it can be driven by a native `<animate>` element.
  `"transform"` is special-cased: `fn` must return a structured op
  `(kind, args)` instead of a plain value, e.g.
  `("translate", (x, y))`, `("rotate", (deg, cx, cy))`, or
  `("scale", (sx, sy))`, since `<animateTransform>` needs a fixed
  `type` plus bare numeric args, not a CSS transform string. A given
  `.transform` binding must return the same `kind` on every state it's
  evaluated on. Two further pseudo-properties are handled specially and
  always apply instantly (no `.transform`-style structure, no
  animation): `"text"` sets `el.textContent`, and `"attr:name"` sets the
  raw SVG attribute `name` via `setAttribute` (for things not otherwise
  reachable, like `d`).
- `fn` is `state -> value`, called in **Python** whenever a transition is
  displayed. There's no `method` to pick anymore: `anim_algebra.svg_diff`
  infers tween vs. snap per property, per transition, by comparing the
  rendered frame before and after. JS never sees `fn` — only the fully
  rendered SVG frame is synced.

`del widget[key]` removes a binding; `key in widget` checks for one;
`widget[key]` returns back the `fn` you assigned. `w.snapshot()` carries
all of a widget's bindings over to the fork.

If several bindings on the *same element* change together (e.g. an
indicator light's `fill`, `fill-opacity`, and `stroke-opacity` all
tweening on together), just give each its own `widget[key] = fn` entry:
`svg_diff` gives each its own independent `<animate>` element, and since
they all start from the same state change they play in sync without
needing to be grouped into one animation.

Because every visible change goes through `_recipe`, computed in Python,
`_repr_mimebundle_` (see `_static_svg()`) doesn't just bake in whatever's
on screen right now: it composes the *whole* displayed trace
(`w.history[0]` through the cursor, via `anim_algebra.seq`) and renders it
with `anim_algebra.compile_to_svg` as native SVG animation elements. A
static export (PDF, a non-JS HTML render, opening the SVG file directly)
replays the recorded scenario on its own, no Jupyter kernel or JS widget
required.
